In [1]:
import json
import pandas as pd
from pathlib import Path
metadata_path = Path("metadata.repository.2026-07-09.json")

with open(metadata_path) as f:
    metadata = json.load(f)

len(metadata), type(metadata)

(1231, list)

In [2]:
rec = metadata[0]

file_id = rec["file_id"]
file_name = rec["file_name"]

entity = rec["associated_entities"][0]
aliquot_barcode = entity["entity_submitter_id"]

patient_barcode = "-".join(aliquot_barcode.split("-")[:3])
sample_barcode = "-".join(aliquot_barcode.split("-")[:4])
sample_type_code = aliquot_barcode.split("-")[3][:2]

file_id, file_name, aliquot_barcode, patient_barcode, sample_barcode, sample_type_code

('744a6d3d-b666-49aa-8d26-47f34e3d1eb5',
 '94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.augmented_star_gene_counts.tsv',
 'TCGA-BH-A18H-01A-11R-A12D-07',
 'TCGA-BH-A18H',
 'TCGA-BH-A18H-01A',
 '01')

In [3]:
if sample_type_code == "01":
    sample_group = "tumor"
elif sample_type_code == "11":
    sample_group = "normal"
else:
    sample_group = "other"

sample_group

'tumor'

In [8]:
rows = []

for rec in metadata:
    file_id = rec["file_id"]
    file_name = rec["file_name"]

    entity = rec["associated_entities"][0]
    aliquot_barcode = entity["entity_submitter_id"]

    patient_barcode = "-".join(aliquot_barcode.split("-")[:3])
    sample_barcode = "-".join(aliquot_barcode.split("-")[:4])
    sample_type_code = aliquot_barcode.split("-")[3][:2]

    if sample_type_code == "01":
        sample_group = "tumor"
    elif sample_type_code == "11":
        sample_group = "normal"
    else:
        sample_group = "other"
        
    row = {
        "file_id": file_id,
        "file_name": file_name,
        "aliquot_barcode": aliquot_barcode,
        "patient_barcode": patient_barcode,
        "sample_barcode": sample_barcode,
        "sample_type_code": sample_type_code,
        "sample_group": sample_group
    }
    
    rows.append(row)

sample_table = pd.DataFrame(rows)

In [9]:
sample_table.head()

,file_id,file_name,aliquot_barcode,patient_barcode,sample_barcode,sample_type_code,sample_group
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,TCGA-BH-A18H-01A-11R-A12D-07,TCGA-BH-A18H,TCGA-BH-A18H-01A,01,tumor
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,TCGA-E2-A14P-01A-31R-A12D-07,TCGA-E2-A14P,TCGA-E2-A14P-01A,01,tumor
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,TCGA-AN-A04A-01A-21R-A034-07,TCGA-AN-A04A,TCGA-AN-A04A-01A,01,tumor
3,2d5b0962-b5c4-4482-9f28-47e4dcdb6df6,602e3d24-a45d-4fe4-9b70-a679d93f470d.rna_seq.a...,TCGA-5L-AAT1-01A-12R-A41B-07,TCGA-5L-AAT1,TCGA-5L-AAT1-01A,01,tumor
4,84cff855-d17a-4c09-b432-53893ae69c1e,03d891b3-8faf-4384-94ce-2015f1ca5df0.rna_seq.a...,TCGA-GI-A2C8-11A-22R-A16F-07,TCGA-GI-A2C8,TCGA-GI-A2C8-11A,11,normal


In [10]:
sample_table.shape, sample_table["sample_group"].value_counts()

((1231, 7),
 sample_group
 tumor     1111
 normal     113
 other        7
 Name: count, dtype: int64)

In [11]:
sample_table.loc[
    sample_table["sample_group"] == "other",
    ["aliquot_barcode", "sample_barcode", "sample_type_code", "file_name"]
]

,aliquot_barcode,sample_barcode,sample_type_code,file_name
199,TCGA-BH-A1FE-06A-11R-A213-07,TCGA-BH-A1FE-06A,06,2e1410fb-4799-4a85-97e2-0d05c9255977.rna_seq.a...
224,TCGA-BH-A1ES-06A-12R-A24H-07,TCGA-BH-A1ES-06A,06,12d9d0ec-4e76-4eba-81b1-6dec7b6f2268.rna_seq.a...
228,TCGA-E2-A15A-06A-11R-A12D-07,TCGA-E2-A15A-06A,06,a6752916-4437-40a7-bc79-d21f52ad1f41.rna_seq.a...
562,TCGA-E2-A15E-06A-11R-A12D-07,TCGA-E2-A15E-06A,06,343a3d35-e60e-4326-bbf8-2bf722c22ccc.rna_seq.a...
664,TCGA-E2-A15K-06A-11R-A12P-07,TCGA-E2-A15K-06A,06,24747d11-c715-48f3-821b-f6dbd6a890fb.rna_seq.a...
1208,TCGA-AC-A6IX-06A-11R-A32P-07,TCGA-AC-A6IX-06A,06,18294571-5fc3-4e68-8b31-5fc6a207639f.rna_seq.a...
1221,TCGA-BH-A18V-06A-11R-A213-07,TCGA-BH-A18V-06A,06,cc3ac2f0-9d26-407d-a8ff-d78170062983.rna_seq.a...


In [12]:
sample_table_clean = sample_table[
    sample_table["sample_type_code"].isin(["01", "11"])
].copy()

In [16]:
print("Sample groups:")
print(sample_table_clean["sample_group"].value_counts())

print("\nRows, columns:", sample_table_clean.shape)
print("Unique patients:", sample_table_clean["patient_barcode"].nunique())
print("Unique samples:", sample_table_clean["sample_barcode"].nunique())

Sample groups:
sample_group
tumor     1111
normal     113
Name: count, dtype: int64

Rows, columns: (1224, 7)
Unique patients: 1095
Unique samples: 1219


In [17]:
sample_table_clean["sample_barcode"].duplicated().sum()

5

In [18]:
dupes = sample_table_clean[
    sample_table_clean["sample_barcode"].duplicated(keep=False)
].sort_values("sample_barcode")

dupes

,file_id,file_name,aliquot_barcode,patient_barcode,sample_barcode,sample_type_code,sample_group
830,6efd37a6-1669-4cdd-95b6-60c688e1e236,304b7293-0a76-4028-9a18-58c87c751ceb.rna_seq.a...,TCGA-A7-A0DB-01A-11R-A277-07,TCGA-A7-A0DB,TCGA-A7-A0DB-01A,01,tumor
831,a8898537-0afa-4655-b123-78c11b90c53a,b4aad532-f92b-43d9-820c-f3c98e382225.rna_seq.a...,TCGA-A7-A0DB-01A-11R-A00Z-07,TCGA-A7-A0DB,TCGA-A7-A0DB-01A,01,tumor
853,50c7c034-fccb-4eb1-b8db-e0a5d9e50172,d820ac02-a30e-437b-b20a-3e92705b7b81.rna_seq.a...,TCGA-A7-A13D-01A-13R-A12P-07,TCGA-A7-A13D,TCGA-A7-A13D-01A,01,tumor
1005,25352bc8-c94c-4542-a7da-d4423adf72cf,1e742dad-8e44-452a-bf2b-9b8ae1ed8bb2.rna_seq.a...,TCGA-A7-A13D-01A-13R-A277-07,TCGA-A7-A13D,TCGA-A7-A13D-01A,01,tumor
377,0a511373-8fd3-433a-a5e1-877530d6a239,5991af25-0b64-43d2-867a-6b9713b17af9.rna_seq.a...,TCGA-A7-A13E-01A-11R-A12P-07,TCGA-A7-A13E,TCGA-A7-A13E-01A,01,tumor
482,4b96ee43-874c-4fa6-b5ce-08caf37ef2e1,d27304e0-1b0b-4fc3-8e0d-af2c305d30f8.rna_seq.a...,TCGA-A7-A13E-01A-11R-A277-07,TCGA-A7-A13E,TCGA-A7-A13E-01A,01,tumor
605,89409cf3-e710-47fd-af3c-6f8f0dec7c90,e6bad6ec-c178-4684-99e8-2504781a022b.rna_seq.a...,TCGA-A7-A26E-01A-11R-A277-07,TCGA-A7-A26E,TCGA-A7-A26E-01A,01,tumor
1073,2b519fff-dd2a-458e-ac2b-573167aeb7f2,a02fb212-a6cc-40b2-9d31-481fc1ce0911.rna_seq.a...,TCGA-A7-A26E-01A-11R-A169-07,TCGA-A7-A26E,TCGA-A7-A26E-01A,01,tumor
106,23ac8b84-f000-45ea-ad35-a46323d3670a,df6189e5-bd61-4be2-9d31-efef56cb4739.rna_seq.a...,TCGA-A7-A26J-01A-11R-A169-07,TCGA-A7-A26J,TCGA-A7-A26J-01A,01,tumor
1216,e5363f56-84fa-4b66-a005-1e7cdcb84c2f,7d787768-efb1-462d-b5aa-5cbf6fc543d4.rna_seq.a...,TCGA-A7-A26J-01A-11R-A277-07,TCGA-A7-A26J,TCGA-A7-A26J-01A,01,tumor


In [20]:
sample_table_dedup = (
    sample_table_clean
    .sort_values(["sample_barcode", "aliquot_barcode"])
    .drop_duplicates(subset="sample_barcode", keep="first")
    .copy()
)

In [21]:
print("Rows, columns:", sample_table_dedup.shape)
print("Unique samples:", sample_table_dedup["sample_barcode"].nunique())
print("Duplicated samples:", sample_table_dedup["sample_barcode"].duplicated().sum())

print("\nSample groups:")
print(sample_table_dedup["sample_group"].value_counts())

Rows, columns: (1219, 7)
Unique samples: 1219
Duplicated samples: 0

Sample groups:
sample_group
tumor     1106
normal     113
Name: count, dtype: int64


In [22]:
pair_status = (
    sample_table_dedup
    .groupby("patient_barcode")["sample_group"]
    .agg(lambda x: set(x))
)

pair_status = (
    sample_table_dedup
    .groupby("patient_barcode")["sample_group"]
    .agg(lambda x: set(x))
)

matched_patients = pair_status[
    pair_status.apply(lambda x: {"tumor", "normal"}.issubset(x))
].index

len(matched_patients)

113

In [23]:
sample_table_dedup["has_matched_pair"] = sample_table_dedup["patient_barcode"].isin(matched_patients)

In [24]:
print("Matched patients:", len(matched_patients))

print(sample_table_dedup.groupby(["has_matched_pair", "sample_group"]).size())

Matched patients: 113
has_matched_pair  sample_group
False             tumor           989
True              normal          113
                  tumor           117
dtype: int64


In [25]:
matched_table = sample_table_dedup[
    sample_table_dedup["patient_barcode"].isin(matched_patients)
].copy()

matched_counts = (
    matched_table
    .groupby(["patient_barcode", "sample_group"])
    .size()
    .unstack(fill_value=0)
)

matched_counts[matched_counts["tumor"] > 1]

sample_group,normal,tumor
patient_barcode,,
TCGA-A7-A0DB,1,2
TCGA-A7-A0DC,1,2
TCGA-A7-A13E,1,2
TCGA-A7-A13G,1,2


In [26]:
multi_tumor_patients = matched_counts[matched_counts["tumor"] > 1].index

sample_table_dedup[
    sample_table_dedup["patient_barcode"].isin(multi_tumor_patients)
].sort_values(["patient_barcode", "sample_group", "sample_barcode"])

,file_id,file_name,aliquot_barcode,patient_barcode,sample_barcode,sample_type_code,sample_group,has_matched_pair
828,d7a48283-c113-4745-be6b-553966e6b457,151af32a-7367-4e35-a575-f6da31aa256a.rna_seq.a...,TCGA-A7-A0DB-11A-33R-A089-07,TCGA-A7-A0DB,TCGA-A7-A0DB-11A,11,normal,True
831,a8898537-0afa-4655-b123-78c11b90c53a,b4aad532-f92b-43d9-820c-f3c98e382225.rna_seq.a...,TCGA-A7-A0DB-01A-11R-A00Z-07,TCGA-A7-A0DB,TCGA-A7-A0DB-01A,01,tumor,True
832,c89590e8-8b74-4c05-a022-e21eab5cffdd,9079bad4-9d01-4eb9-999d-35ad734be2e8.rna_seq.a...,TCGA-A7-A0DB-01C-02R-A277-07,TCGA-A7-A0DB,TCGA-A7-A0DB-01C,01,tumor,True
108,9e056a2a-a643-478a-b96c-85d5221f487a,c6a90de3-979b-46b0-9f84-2875c1e38742.rna_seq.a...,TCGA-A7-A0DC-11A-41R-A089-07,TCGA-A7-A0DC,TCGA-A7-A0DC-11A,11,normal,True
110,0e1999e0-9c02-4804-88b6-60b46d100739,db5dab56-838e-4784-a7cf-f05296e356f7.rna_seq.a...,TCGA-A7-A0DC-01A-11R-A00Z-07,TCGA-A7-A0DC,TCGA-A7-A0DC-01A,01,tumor,True
109,3e8cff80-ed17-479c-8327-ca8e71ee7482,0c814cd9-5b2c-4764-9e6a-6ad0403fcd7d.rna_seq.a...,TCGA-A7-A0DC-01B-04R-A22O-07,TCGA-A7-A0DC,TCGA-A7-A0DC-01B,01,tumor,True
276,50260fba-c435-480a-9bb8-f2839ba499a4,1d390e7f-bbbb-45b5-9466-d199f19106b2.rna_seq.a...,TCGA-A7-A13E-11A-61R-A12P-07,TCGA-A7-A13E,TCGA-A7-A13E-11A,11,normal,True
377,0a511373-8fd3-433a-a5e1-877530d6a239,5991af25-0b64-43d2-867a-6b9713b17af9.rna_seq.a...,TCGA-A7-A13E-01A-11R-A12P-07,TCGA-A7-A13E,TCGA-A7-A13E-01A,01,tumor,True
481,06952c59-978d-4ef4-b716-c55dd6747699,e843a939-f1a2-4e98-87b5-69abef893a51.rna_seq.a...,TCGA-A7-A13E-01B-06R-A277-07,TCGA-A7-A13E,TCGA-A7-A13E-01B,01,tumor,True
1107,90617797-d2c7-4c49-bffe-b2bea96a7cf1,cf7f5e39-f761-424f-9987-21b328afa0ae.rna_seq.a...,TCGA-A7-A13G-11A-51R-A13Q-07,TCGA-A7-A13G,TCGA-A7-A13G-11A,11,normal,True


In [31]:
sample_table_dedup["vial"] = (
    sample_table_dedup["sample_barcode"]
    .str.split("-")
    .str[3]
    .str[2:]
)

In [49]:
tumors = sample_table_dedup[sample_table_dedup["sample_group"] == "tumor"].copy()

tumor_one_per_patient = (
    tumors
    .sort_values(["patient_barcode", "vial", "sample_barcode"])
    .drop_duplicates(subset="patient_barcode", keep="first")
    .copy()
)

In [50]:
print("Tumor samples before:", tumors.shape[0])
print("Tumor patients before:", tumors["patient_barcode"].nunique())
print("Tumor samples after:", tumors_one_per_patient.shape[0])
print("Tumor patients after:", tumors_one_per_patient["patient_barcode"].nunique())
print(tumors_one_per_patient["vial"].value_counts())

Tumor samples before: 1106
Tumor patients before: 1095
Tumor samples after: 1095
Tumor patients after: 1095
vial
A    1081
B      14
Name: count, dtype: int64


In [51]:
matched_table.head()

,file_id,file_name,aliquot_barcode,patient_barcode,sample_barcode,sample_type_code,sample_group,has_matched_pair
171,04e03ae1-b2f3-446d-862d-172fea6320de,58b399a0-3070-44aa-9a40-f0e2c0fea0fc.rna_seq.a...,TCGA-A7-A0CE-01A-11R-A00Z-07,TCGA-A7-A0CE,TCGA-A7-A0CE-01A,01,tumor,True
242,23bf74db-bb4e-44c5-8473-e651b818e460,308e15c9-16b0-4f24-b9dc-e9dc91038579.rna_seq.a...,TCGA-A7-A0CE-11A-21R-A089-07,TCGA-A7-A0CE,TCGA-A7-A0CE-11A,11,normal,True
1104,64657d41-17ad-4692-8a13-0cf55c55f11d,6f73500d-4b23-4f14-911b-fbc69bd38f4b.rna_seq.a...,TCGA-A7-A0CH-01A-21R-A00Z-07,TCGA-A7-A0CH,TCGA-A7-A0CH-01A,01,tumor,True
236,b9bdb9d3-4d2b-4a53-9804-d0511174e553,49e456c9-c47b-4663-bf6f-9f8f4e4e71a5.rna_seq.a...,TCGA-A7-A0CH-11A-32R-A089-07,TCGA-A7-A0CH,TCGA-A7-A0CH-11A,11,normal,True
658,69aebf68-5357-4269-84fb-ee91a6bd67fd,8f4d89ea-3207-4fa7-9898-9b78cb971577.rna_seq.a...,TCGA-A7-A0D9-01A-31R-A056-07,TCGA-A7-A0D9,TCGA-A7-A0D9-01A,01,tumor,True


In [52]:
matched_single_vial = (
    sample_table_dedup[sample_table_dedup["patient_barcode"].isin(matched_patients)]
    .sort_values(["patient_barcode", "sample_group", "vial"])
    .drop_duplicates(subset=["patient_barcode", "sample_group"], keep="first")
    .copy()
)

In [53]:
print(matched_single_vial["sample_group"].value_counts())
print(matched_single_vial["patient_barcode"].nunique())
print(matched_single_vial.shape)

sample_group
normal    113
tumor     113
Name: count, dtype: int64
113
(226, 9)


In [62]:
metadata_dir = Path("data/metadata")
metadata_dir.mkdir(parents=True, exist_ok=True)

sample_table.to_csv(
    metadata_dir / "tcga_brca_star_counts_all_parsed.csv",
    index=False
)

sample_table_clean.to_csv(
    metadata_dir / "tcga_brca_primary_tumor_solid_normal_clean.csv",
    index=False
)

sample_table_dedup.to_csv(
    metadata_dir / "tcga_brca_sample_level_dedup.csv",
    index=False
)

tumor_one_per_patient.to_csv(
    metadata_dir / "tcga_brca_tumor_one_per_patient.csv",
    index=False
)

matched_single_vial.to_csv(
    metadata_dir / "tcga_brca_matched_tumor_normal_one_per_group.csv",
    index=False
)

In [63]:
list(metadata_dir.glob("*.csv"))

[PosixPath('data/metadata/tcga_brca_star_counts_all_parsed.csv'),
 PosixPath('data/metadata/tcga_brca_matched_tumor_normal_one_per_group.csv'),
 PosixPath('data/metadata/tcga_brca_tumor_one_per_patient.csv'),
 PosixPath('data/metadata/tcga_brca_sample_level_dedup.csv'),
 PosixPath('data/metadata/tcga_brca_primary_tumor_solid_normal_clean.csv')]